In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# import sys
# import os
# # path = "/content/drive/MyDrive/Colab Notebooks/rf_cnn"
# path = "~/Documents/pj_cnn"
# os.chdir(path)
# sys.path.append(path)

FileNotFoundError: [Errno 2] No such file or directory: '~/Documents/pj_cnn'

In [ ]:
# @title
# !unzip "data.zip" -d "."

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
  inflating: ./data/images/Rect/signal_snr05_doppler300_img00220.png  
  inflating: ./data/images/Rect/signal_snr03_doppler070_img02369.png  
  inflating: ./data/images/Rect/signal_snr01_doppler004_img03404.png  
  inflating: ./data/images/Rect/signal_snr08_doppler070_img02993.png  
  inflating: ./data/images/Rect/signal_snr01_doppler300_img03898.png  
  inflating: ./data/images/Rect/signal_snr03_doppler000_img00372.png  
  inflating: ./data/images/Rect/signal_snr01_doppler004_img03555.png  
  inflating: ./data/images/Rect/signal_snr01_doppler004_img00387.png  
  inflating: ./data/images/Rect/signal_snr08_doppler300_img01180.png  
  inflating: ./data/images/Rect/signal_snr04_doppler004_img01222.png  
  inflating: ./data/images/Rect/signal_snr02_doppler070_img01119.png  
  inflating: ./data/images/Rect/signal_snr05_doppler070_img00631.png  
  inflating: ./data/images/Rect/signal_snr05_doppler004_img01953.png  
  inflating: ./data/

In [ ]:
# !ls

sample_data


In [3]:
!pip install -r "requirements.txt"

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [1]:
from dataset.rf_dataset import get_dataloaders
from models.cnn_model import RFNet
from train.train import train_one_epoch, validation, save_checkpoint, load_checkpoint
from utils.check_dataset import check_dataset, download_dataset
from config.config import *
import torch
import os

/home/nhan/miniconda3/envs/py312/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# @title
def check_device():
    is_cuda = torch.cuda.is_available()
    if is_cuda:
        print(f"- Sử dụng GPU: {torch.cuda.get_device_name(0)}")
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
        print("- Sử dụng CPU")
    return device

def get_version(path):
    if not os.path.exists(path):
        os.makedirs(path)
    versions = [d for d in os.listdir(path) if d.startswith("ver")]
    ver = len(versions) + 1
    return f"ver{ver}"




In [4]:

def main(resume=False, checkpoint_path=None):
    # prepare
    if checkpoint_path is None:
      checkpoint_path = os.path.join(CHECKPOINT_DIR, get_version(CHECKPOINT_DIR))
      os.mkdir(checkpoint_path)
      print(f"- Lượt huấn luyện mới lưu tại: {checkpoint_path}")
    download_dataset(DATASET_PATH, KAGGLE_PATH)
    check_dataset(DATASET_PATH)
    device = check_device()
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True


    # dataset
    train_loader, val_loader = get_dataloaders(DATASET_PATH, BATCH_SIZE, device, NUM_WORKERS, TRAIN_SPLIT, SEED)

    # model
    model = RFNet(num_classes=NUM_CLASSES)
    model.to(device)

    # loss với label smoothing
    criterion = torch.nn.CrossEntropyLoss(label_smoothing=0.1)

    # optimizer
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY
    )

    # learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.3,
        patience=2 # giảm lr nếu val loss không cải thiện sau 2 epochs
    )

    best_acc = 0.0

    start_epoch = 0
    if resume:
        start_epoch = load_checkpoint(
            model,
            optimizer,
            f"{checkpoint_path}/last_checkpoint.pth",
            device
        )
        print(f"- Đã load checkpoint từ {checkpoint_path}: epoch {start_epoch+1}")

    for epoch in range(start_epoch, EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        train_loss, train_acc = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )
        val_loss, val_acc = validation(
            model,
            val_loader,
            criterion,
            device
        )
        scheduler.step(val_loss)
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}")
        print("LR: ", optimizer.param_groups[0]["lr"])

        save_checkpoint(
            model,
            optimizer,
            epoch + 1,
            f"{checkpoint_path}/last_checkpoint.pth"
        )
        print("- Đã lưu check_point")
        if (epoch + 1) % SAVE_EVERY == 0:

            save_checkpoint(
                model,
                optimizer,
                epoch + 1,
                f"{checkpoint_path}/checkpoint_epoch_{epoch+1}.pth"
            )

        # save best model
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f"{checkpoint_path}/{MODEL_NAME}")
            print(f"- Model saved with val acc: {best_acc:.4f}")

In [7]:
if __name__ == "__main__":
    main(resume=True, checkpoint_path="checkpoints/ver9")
    # main()

- Dataset đã tồn tại, không cần tải lại.
- Số lượng lớp đầu ra: 12
	0: "Barker"		-> 6400 ảnh
	1: "BPSK"		-> 6400 ảnh
	2: "16-QAM"		-> 6400 ảnh
	3: "StepFM"		-> 6400 ảnh
	4: "B-FM"		-> 6400 ảnh
	5: "PAM4"		-> 6400 ảnh
	6: "LFM"		-> 6400 ảnh
	7: "QPSK"		-> 6400 ảnh
	8: "CPFSK"		-> 6400 ảnh
	9: "DSB-AM"		-> 6400 ảnh
	10: "GFSK"		-> 6400 ảnh
	11: "Rect"		-> 6400 ảnh
- Tổng số ảnh trong dataset: 76800
- Kích thước ảnh mẫu:  (224, 224)
- Sử dụng CPU
- Đã load checkpoint từ checkpoints/ver9: epoch 6

Epoch 6/20


  0%|          | 1/1920 [00:05<2:54:32,  5.46s/it]


KeyboardInterrupt: 